# Hyperparameter optimization: `lr_find`

Picking a learning rate is the most consequential hyperparameter
decision in most training runs. idris-ml ships fastai's `lr_find`
(LR-range test) as a quick screening tool: train for ~100 iterations
with LR sweeping log-uniformly from very small (1e-7) to very large
(10), record the smoothed loss at each LR, and recommend an LR.

The API lives in the **Hpo** module; `Schedule` has the LR schedules
(`oneCycle`, `cosineAnnealing`, etc.) to feed the recommendation into.


## API surface

`lrFind` takes a config, an epoch function, a data source, the native
optimizer, and the model. It returns the loss curve plus a recommended
LR.


In [ ]:
:t lrFind


In [ ]:
:t LrFindConfig


In [ ]:
:t LrFindResult


In [ ]:
:t defaultLrFindConfig


The defaults match fastai: 100 iterations across LR `1e-7..10`,
EMA-smoothed loss with β=0.98, recommended LR = (LR at steepest
negative slope) ÷ 10.


## Running it on a real example

The notebook kernel buffers all output, so the per-iteration `iter\tlr\tloss\tsmoothed`
log lines from a 100-iter sweep print all-at-once at the end — not great
for interactive use. Run `lr_find` from the compiled examples instead:

```bash
make example-supervised SUPERVISED_ARGS="--lr-find 1"
make example-rnn        RNN_ARGS="--lr-find 1"
```

Each finishes in ~1 second on tape; output streams line-by-line so you
can watch the loss curve develop.


## Cross-backend agreement gate

`lr_find` is a screening tool, not a recommendation engine. The
fastai heuristic ("steepest descent ÷ 10") can latch onto noisy
local-slope features when the loss curve is essentially flat —
typically when the architecture is too small to clearly benefit
from any LR.

idris-ml's discipline: run `lr_find` on **both** the
Idris example and its PyTorch reference, and only trust the
recommendation when the two backends agree within **2×**. See
`docs/develop/hyperparameter-tuning-2026.md`:

| Example | Idris rec | PyTorch rec | Ratio | Decision |
|---|---|---|---|---|
| Supervised | 0.020 | 0.024 | 1.20× | actionable (within 2×) |
| Rnn | 0.572 | 0.0015 | 380× | unreliable, don't change default |

When the gate fails, leave the default unchanged: both backends
are picking up flat-curve noise, not a real signal.


## Applying the recommendation

Once you have a recommended LR, plug it into a fastai-style schedule
(typically `oneCycle`) and attach the schedule to the optimizer with
`withSchedule`. The `fit` driver ticks the schedule once per epoch.


In [ ]:
:t withSchedule


In [ ]:
:t oneCycle


Wire-up sketch:

```idris
opt <- adam 0.02 defaultOpts                     -- lr from the recommendation
let sched = oneCycle 0.02 25.0 1.0e5 0.25 1000   -- lrMax=0.02, fastai defaults
(MkBang (epochs, loss) # trained) <-
  fitSupervised (withSchedule sched opt) lossFn stream (simpleConfig 1000) model
```

`fit` calls `tick opt epoch` before every epoch; with a schedule attached,
`tick` pushes `sched epoch` into the optimizer's base LR, so the LR ramps
linearly through warmup and then cosine-anneals down. For loops that don't
go through `fit`, `applySchedule` is the manual form of the same binding:
it wires `setLearningRate opt (sched epoch)` into `TrainConfig.beforeEpoch`.


## Further reading

- `docs/develop/hyperparameter-tuning-2026.md` — per-example
  dogfooding log with cross-backend agreement ratios and decisions.
- `docs/develop/example-coverage.md` — what's covered in the
  example suite + which problems are wrong-shape.
- `Schedule.idr` — full schedule API (`oneCycle`,
  `cosineAnnealing`, `withWarmup`, `cosineWithWarmup`, `stepLR`,
  `exponentialLR`).
